In [ ]:
# Resolve public-release paths without requiring internal project documentation.
from pathlib import Path as _ReleasePath
import os as _release_os
_release_root = next((p for p in (_ReleasePath.cwd(), *_ReleasePath.cwd().parents)
                      if (p / ".shap_multiplicity_root").is_file()), None)
if _release_root is None:
    raise FileNotFoundError("Launch Jupyter within this repository.")
_release_os.chdir(_release_root)

import os, pickle, itertools, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib, matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import scienceplots
    plt.style.use(['science', 'no-latex'])
except ImportError:
    print('scienceplots not installed; using default style')

from plot_violin import compute_jaccard_distance

matplotlib.rcParams.update({
    'font.size':       12,
    'axes.labelsize':  13,
    'axes.titlesize':  13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.titlesize':14,
})

DATASETS      = ['german', 'diabetes', 'acs', 'gmsc']
MODELS        = ['mlp', 'ftt']
METHODS       = ['random', 'kmeans', 'pred_stratified', 'cte']
METHOD_LABELS = {'random': 'Random', 'kmeans': 'KMeans',
                 'pred_stratified': 'PredStrat', 'cte': 'CTE'}
METHOD_COLORS = {'random': '#4393C3', 'kmeans': '#D6604D',
                 'pred_stratified': '#56A65A', 'cte': '#9970AB'}
DATASET_LABELS = {'german': 'German', 'diabetes': 'Diabetes', 'acs': 'ACS', 'gmsc': 'GMSC'}
N_FEATS       = {'german': 16, 'diabetes': 8, 'acs': 8, 'gmsc': 10, 'arrhythmia': 278}

JAC_K      = 3
N_BG_SEEDS = 5
JAC_VALUES = [0.0, 0.5, 0.8, 1.0]
JAC_COLORS = {0.0: '#2b8a3e', 0.5: '#ffd43b', 0.8: '#fd7e14', 1.0: '#e03131'}
JAC_LABELS = {0.0: 'Jac=0 (no change)', 0.5: 'Jac=0.5 (1 feature)',
              0.8: 'Jac=0.8 (2 features)', 1.0: 'Jac=1.0 (all different)'}
PAIRS_10   = list(itertools.combinations(range(N_BG_SEEDS), 2))

RES_DIR = 'results_tree'
FIG_DIR = 'figures_b/churn_analysis'
os.makedirs(FIG_DIR, exist_ok=True)

print('Config loaded.')

In [ ]:
import glob, re

def load_sv(ds, model, method, s_idx):
    # 1) Prefer canonical chunked files (results_tree/{ds}_{model}_0_0_{method}_s{s_idx}_c*_sv.pkl)
    #    These are the authoritative re-run files for large datasets (GMSC, ACS).
    pattern = os.path.join(RES_DIR, f'{ds}_{model}_0_0_{method}_s{s_idx}_c*_sv.pkl')
    chunk_paths = sorted(
        glob.glob(pattern),
        key=lambda p: int(re.search(r'_c(\d+)_sv', p).group(1))
    )
    if chunk_paths:
        arrays = []
        for cp in chunk_paths:
            with open(cp, 'rb') as f:
                obj = pickle.load(f)
            arrays.append(obj.values if hasattr(obj, 'values') else obj['values'])
        return np.concatenate(arrays, axis=0)

    # 2) Fall back to non-chunked file (small datasets: German, Diabetes)
    fn   = f'{ds}_{model}_0_0_{method}_s{s_idx}_sv.pkl'
    path = os.path.join(RES_DIR, fn)
    if os.path.exists(path):
        with open(path, 'rb') as f:
            obj = pickle.load(f)
        return obj.values if hasattr(obj, 'values') else obj['values']

    return None

print('Load helper ready (chunked files take priority over non-chunked).')

In [ ]:
# sv_data[ds][model][method] = list of 5 arrays (N_test, F), or None if any seed missing
sv_data = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}

for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = [load_sv(ds, model, method, s) for s in range(N_BG_SEEDS)]
            sv_data[ds][model][method] = None if any(v is None for v in svs) else svs
        avail = [m for m in METHODS if sv_data[ds][model][m] is not None]
        print(f'{ds}/{model}: {avail}')

print('\nData loaded.')

In [ ]:
# Discrete Jaccard proportions and top-1 churn at bg_size=100

def round_jac(val):
    return min(JAC_VALUES, key=lambda x: abs(x - val))

jac_props  = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
top1_churn = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}

for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue

            min_n = min(len(v) for v in svs)

            # --- discrete Jaccard ---
            all_jac = np.concatenate([
                compute_jaccard_distance(svs[i][:min_n], svs[j][:min_n], k=JAC_K)
                for i, j in PAIRS_10
            ])
            counts = {v: 0 for v in JAC_VALUES}
            for val in all_jac:
                counts[round_jac(float(val))] += 1
            total = len(all_jac)
            jac_props[ds][model][method] = {v: counts[v] / total for v in JAC_VALUES}

            # --- top-1 churn ---
            top1s = [np.argmax(np.abs(svs[s][:min_n]), axis=1) for s in range(N_BG_SEEDS)]
            churn_arr = np.concatenate([(top1s[i] != top1s[j]).astype(float) for i, j in PAIRS_10])
            top1_churn[ds][model][method] = float(churn_arr.mean())

print('Jaccard proportions and top-1 churn computed.')

In [ ]:
# Discrete Jaccard stacked bar charts — 1 figure per dataset (1×2: mlp, ftt)

for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Discrete Top-3 Jaccard Distribution  (bg_size=100)',
        fontsize=13, fontweight='bold'
    )

    for ax, model in zip(axes, MODELS):
        avail = [m for m in METHODS if m in jac_props[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue

        x       = np.arange(len(avail))
        bottoms = np.zeros(len(avail))

        for jv in JAC_VALUES:
            heights = np.array([jac_props[ds][model][m][jv] for m in avail])
            ax.bar(x, heights, bottom=bottoms,
                   color=JAC_COLORS[jv], edgecolor='white', linewidth=0.5)
            for xi, (h, b) in enumerate(zip(heights, bottoms)):
                if h > 0.04:
                    txt_color = 'white' if jv == 1.0 else 'black'
                    ax.text(xi, b + h / 2, f'{h:.2%}',
                            ha='center', va='center', fontsize=8,
                            color=txt_color, fontweight='bold')
            bottoms += heights

        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, 1.02)
        ax.set_ylabel('Proportion of (pair, instance) observations')
        ax.set_title(model.upper())

    handles = [mpatches.Patch(color=JAC_COLORS[jv], label=JAC_LABELS[jv]) for jv in JAC_VALUES]
    fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=9,
               bbox_to_anchor=(0.5, -0.06), framealpha=0.9)

    fig.tight_layout(rect=[0, 0.10, 1, 1])
    fname = os.path.join(FIG_DIR, f'churn_jaccard_stacked_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Top-1 churn bar charts — 1 figure per dataset (1×2: mlp, ftt)

for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Top-1 Feature Churn  (bg_size=100)',
        fontsize=13, fontweight='bold'
    )

    for ax, model in zip(axes, MODELS):
        avail  = [m for m in METHODS if m in top1_churn[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue

        x      = np.arange(len(avail))
        values = [top1_churn[ds][model][m] for m in avail]
        colors = [METHOD_COLORS[m] for m in avail]

        ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5)
        for xi, v in enumerate(values):
            ax.text(xi, v + 0.005, f'{v:.1%}', ha='center', va='bottom', fontsize=9)

        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, max(values) * 1.30 + 0.01)
        ax.set_ylabel('Fraction of pair-instance comparisons with Top-1 change')
        ax.set_title(model.upper())

    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f'churn_top1_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Top-1 any-seed churn: % of test instances where at least 1 seed disagrees on top-1 feature

top1_anychurn = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}

for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue
            min_n = min(len(v) for v in svs)
            # top1s[s, i] = top-1 feature index for instance i in seed s
            top1s = np.stack([np.argmax(np.abs(svs[s][:min_n]), axis=1)
                              for s in range(N_BG_SEEDS)], axis=0)  # (5, N)
            all_agree = np.all(top1s == top1s[[0], :], axis=0)      # (N,) True if all 5 seeds agree
            top1_anychurn[ds][model][method] = float((~all_agree).mean())

print('Any-seed top-1 churn computed.')

# ── Plot ──────────────────────────────────────────────────────────────────────
for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Top-1 Feature Churn: Any-Seed  (bg_size=100)\n'
        r'(% of instances where $\geq$1 seed disagrees on top-1 feature)',
        fontsize=12, fontweight='bold'
    )

    for ax, model in zip(axes, MODELS):
        avail  = [m for m in METHODS if m in top1_anychurn[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue

        x      = np.arange(len(avail))
        values = [top1_anychurn[ds][model][m] for m in avail]
        colors = [METHOD_COLORS[m] for m in avail]

        ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5)
        for xi, v in enumerate(values):
            ax.text(xi, v + 0.005, f'{v:.2%}', ha='center', va='bottom', fontsize=9)

        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, max(values) * 1.30 + 0.01)
        ax.set_ylabel('% of instances where top-1 changes')
        ax.set_title(model.upper())

    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f'churn_top1_anyseed_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Rank position of consensus top-3 features
# consensus_top3[i][c] = c-th most important feature for instance i (by mean |SHAP| across seeds)
# rank_dist[ds][model][method][c] = 1D array of rank positions (1-indexed) across all (instance, seed)

rank_dist = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}

for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue

            min_n   = min(len(v) for v in svs)
            abs_svs = np.stack([np.abs(v[:min_n]) for v in svs], axis=0)  # (5, N, F)

            mean_abs      = abs_svs.mean(axis=0)                         # (N, F)
            consensus_top3 = np.argsort(-mean_abs, axis=1)[:, :3]        # (N, 3)

            # ranks[s, i, f] = rank of feature f for instance i in seed s (0-indexed, 0=most important)
            ranks = np.argsort(np.argsort(-abs_svs, axis=2), axis=2)     # (5, N, F)

            rank_dist[ds][model][method] = {}
            for c in range(3):
                feat_idx = consensus_top3[:, c]       # (N,)
                rpos = np.array([
                    ranks[s, np.arange(min_n), feat_idx]
                    for s in range(N_BG_SEEDS)
                ]).flatten()                           # (5×N,)
                rank_dist[ds][model][method][c] = rpos + 1   # 1-indexed

print('Rank distributions computed.')

In [ ]:
# Rank position heatmaps — 1 figure per (dataset, model), 1×4 subplots (methods)

for ds in DATASETS:
    d = N_FEATS[ds]

    for model in MODELS:
        fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
        fig.suptitle(
            f'{DATASET_LABELS[ds]}  ·  {model.upper()}  —  Rank Positions of Consensus Top-3 Features  (bg_size=100)',
            fontsize=12, fontweight='bold'
        )

        for col, method in enumerate(METHODS):
            ax = axes[col]
            rd = rank_dist[ds][model].get(method)

            if rd is None:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                        transform=ax.transAxes, fontsize=12, color='gray')
                ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(METHOD_LABELS[method], fontsize=11)
                continue

            # Build 3 × d heatmap  (rows = consensus rank, cols = rank in individual seed)
            heatmap = np.zeros((3, d))
            for c in range(3):
                rpos  = rd[c]
                total = len(rpos)
                for r in range(1, d + 1):
                    heatmap[c, r - 1] = np.sum(rpos == r) / total

            im = ax.imshow(heatmap, aspect='auto', cmap='YlOrRd',
                           vmin=0, vmax=min(heatmap.max(), 1.0))

            for ci in range(3):
                for ri in range(d):
                    val = heatmap[ci, ri]
                    if val > 0.05:
                        txt_color = 'white' if val > 0.55 else 'black'
                        ax.text(ri, ci, f'{val:.0%}', ha='center', va='center',
                                fontsize=7, color=txt_color)

            ax.set_xticks(range(d))
            ax.set_xticklabels([str(r + 1) for r in range(d)], fontsize=8)
            ax.set_yticks([0, 1, 2])
            ax.set_yticklabels(['Avg #1', 'Avg #2', 'Avg #3'], fontsize=9)
            ax.set_xlabel('Rank in individual seed', fontsize=9)
            ax.set_title(METHOD_LABELS[method], fontsize=11)

        fig.tight_layout()
        fname = os.path.join(FIG_DIR, f'churn_rankpos_{ds}_{model}.pdf')
        fig.savefig(fname, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')

In [ ]:
# Rank position heatmaps — Random & CTE only, 1 figure per (dataset, model), 1×2 subplots

METHODS_2       = ['random', 'cte']
METHOD_LABELS_2 = {'random': 'Random', 'cte': 'CTE'}

for ds in DATASETS:
    d = N_FEATS[ds]

    for model in MODELS:
        fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
        fig.suptitle(
            f'{DATASET_LABELS[ds]}  ·  {model.upper()}  —  Rank Positions of Consensus Top-3 Features  (bg_size=100)',
            fontsize=12, fontweight='bold'
        )

        for col, method in enumerate(METHODS_2):
            ax = axes[col]
            rd = rank_dist[ds][model].get(method)

            if rd is None:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                        transform=ax.transAxes, fontsize=12, color='gray')
                ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(METHOD_LABELS_2[method], fontsize=12)
                continue

            heatmap = np.zeros((3, d))
            for c in range(3):
                rpos  = rd[c]
                total = len(rpos)
                for r in range(1, d + 1):
                    heatmap[c, r - 1] = np.sum(rpos == r) / total

            ax.imshow(heatmap, aspect='auto', cmap='YlOrRd',
                      vmin=0, vmax=min(heatmap.max(), 1.0))

            for ci in range(3):
                for ri in range(d):
                    val = heatmap[ci, ri]
                    if val > 0.05:
                        txt_color = 'white' if val > 0.55 else 'black'
                        ax.text(ri, ci, f'{val:.0%}', ha='center', va='center',
                                fontsize=8, color=txt_color)

            ax.set_xticks(range(d))
            ax.set_xticklabels([str(r + 1) for r in range(d)], fontsize=9)
            ax.set_yticks([0, 1, 2])
            ax.set_yticklabels(['Avg #1', 'Avg #2', 'Avg #3'], fontsize=10)
            ax.set_xlabel('Rank in individual seed', fontsize=9)
            ax.set_title(METHOD_LABELS_2[method], fontsize=12)

        fig.tight_layout()
        fname = os.path.join(FIG_DIR, f'churn_rankpos2_{ds}_{model}.pdf')
        fig.savefig(fname, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')

## ══════════════ Top-2 Jaccard ══════════════

In [ ]:
# Config and discrete Jaccard proportions for top-2
_jv2 = [0.0, round(2/3, 6), 1.0]
_cmap_obj = plt.cm.RdYlGn_r
_colors2 = {v: _cmap_obj(t) for v, t in zip(_jv2, np.linspace(0.05, 0.95, len(_jv2)))}
_labels2 = {
    0.0:           'Jac=0 (2 feat. overlap)',
    round(2/3, 6): 'Jac=0.667 (1 feat. overlap)',
    1.0:           'Jac=1.0 (0 feat. overlap)',
}

def round_jac_k(val, jac_vals):
    return min(jac_vals, key=lambda x: abs(x - val))

jac_props_k2 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue
            min_n = min(len(v) for v in svs)
            all_jac = np.concatenate([
                compute_jaccard_distance(svs[i][:min_n], svs[j][:min_n], k=2)
                for i, j in PAIRS_10
            ])
            counts = {v: 0 for v in _jv2}
            for val in all_jac:
                counts[round_jac_k(float(val), _jv2)] += 1
            total = len(all_jac)
            jac_props_k2[ds][model][method] = {v: counts[v] / total for v in _jv2}

print('Top-2 Jaccard proportions computed.')

In [ ]:
# Top-2 discrete Jaccard stacked bar charts — 1 figure per dataset (1x2: mlp, ftt)
for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Discrete Top-2 Jaccard Distribution  (bg_size=100)',
        fontsize=13, fontweight='bold'
    )
    for ax, model in zip(axes, MODELS):
        avail = [m for m in METHODS if m in jac_props_k2[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue
        x       = np.arange(len(avail))
        bottoms = np.zeros(len(avail))
        for jv in _jv2:
            heights = np.array([jac_props_k2[ds][model][m][jv] for m in avail])
            ax.bar(x, heights, bottom=bottoms,
                   color=_colors2[jv], edgecolor='white', linewidth=0.5)
            for xi, (h, b) in enumerate(zip(heights, bottoms)):
                if h > 0.04:
                    txt_color = 'white' if jv == _jv2[-1] else 'black'
                    ax.text(xi, b + h / 2, f'{h:.2%}',
                            ha='center', va='center', fontsize=8,
                            color=txt_color, fontweight='bold')
            bottoms += heights
        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, 1.02)
        ax.set_ylabel('Proportion of (pair, instance) observations')
        ax.set_title(model.upper())
    handles = [mpatches.Patch(color=_colors2[jv], label=_labels2[jv]) for jv in _jv2]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9,
               bbox_to_anchor=(0.5, -0.06), framealpha=0.9)
    fig.tight_layout(rect=[0, 0.10, 1, 1])
    fname = os.path.join(FIG_DIR, f'churn_jaccard_k2_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Top-2 pairwise churn: % of (pair, instance) where top-2 SET changes
# Derived directly from jac_props_k2: churn = 1 - prop(Jac=0)
topk_pairchurn_k2 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            if method not in jac_props_k2[ds][model]:
                continue
            topk_pairchurn_k2[ds][model][method] = 1.0 - jac_props_k2[ds][model][method][0.0]

for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Top-2 Pairwise Churn  (bg_size=100)',
        fontsize=13, fontweight='bold'
    )
    for ax, model in zip(axes, MODELS):
        avail = [m for m in METHODS if m in topk_pairchurn_k2[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue
        x      = np.arange(len(avail))
        values = [topk_pairchurn_k2[ds][model][m] for m in avail]
        colors = [METHOD_COLORS[m] for m in avail]
        ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5)
        for xi, v in enumerate(values):
            ax.text(xi, v + 0.005, f'{v:.2%}', ha='center', va='bottom', fontsize=9)
        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, max(values) * 1.30 + 0.01)
        ax.set_ylabel('% of pairs where top-2 set changes')
        ax.set_title(model.upper())
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f'churn_topk_pair_k2_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Top-2 any-seed churn: % of instances where top-2 SET differs across any seed
topk_anychurn_k2 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue
            min_n = min(len(v) for v in svs)
            # Sort feature indices so set comparison is order-independent
            topk_sorted = np.stack([
                np.sort(np.argsort(-np.abs(svs[s][:min_n]), axis=1)[:, :2], axis=1)
                for s in range(N_BG_SEEDS)
            ], axis=0)                                           # (5, N, 2)
            all_agree = np.all(topk_sorted == topk_sorted[[0], :, :], axis=0)  # (N, 2)
            stable = np.all(all_agree, axis=1)                  # (N,)
            topk_anychurn_k2[ds][model][method] = float((~stable).mean())

for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Top-2 Any-Seed Churn  (bg_size=100)\n'
        r'(% instances where $\geq$1 seed has a different top-2 set)',
        fontsize=12, fontweight='bold'
    )
    for ax, model in zip(axes, MODELS):
        avail = [m for m in METHODS if m in topk_anychurn_k2[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue
        x      = np.arange(len(avail))
        values = [topk_anychurn_k2[ds][model][m] for m in avail]
        colors = [METHOD_COLORS[m] for m in avail]
        ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5)
        for xi, v in enumerate(values):
            ax.text(xi, v + 0.005, f'{v:.2%}', ha='center', va='bottom', fontsize=9)
        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, max(values) * 1.30 + 0.01)
        ax.set_ylabel('% of instances where top-2 set changes')
        ax.set_title(model.upper())
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f'churn_topk_any_k2_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Rank position distribution for consensus top-2
rank_dist_k2 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue
            min_n   = min(len(v) for v in svs)
            abs_svs = np.stack([np.abs(v[:min_n]) for v in svs], axis=0)  # (5, N, F)
            mean_abs       = abs_svs.mean(axis=0)
            consensus_topk = np.argsort(-mean_abs, axis=1)[:, :2]         # (N, 2)
            ranks = np.argsort(np.argsort(-abs_svs, axis=2), axis=2)      # (5, N, F)
            rank_dist_k2[ds][model][method] = {}
            for c in range(2):
                feat_idx = consensus_topk[:, c]
                rpos = np.array([
                    ranks[s, np.arange(min_n), feat_idx]
                    for s in range(N_BG_SEEDS)
                ]).flatten()
                rank_dist_k2[ds][model][method][c] = rpos + 1  # 1-indexed
print('Top-2 rank distributions computed.')

In [ ]:
# Top-2 rank position heatmaps — 1 figure per (dataset, model), 1x4 subplots
for ds in DATASETS:
    d = N_FEATS[ds]
    for model in MODELS:
        fig, axes = plt.subplots(1, 4, figsize=(14, 3.0))
        fig.suptitle(
            f'{DATASET_LABELS[ds]}  ·  {model.upper()}  —  Rank Positions of Consensus Top-2 Features  (bg_size=100)',
            fontsize=12, fontweight='bold'
        )
        for col, method in enumerate(METHODS):
            ax = axes[col]
            rd = rank_dist_k2[ds][model].get(method)
            if rd is None:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                        transform=ax.transAxes, fontsize=12, color='gray')
                ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(METHOD_LABELS[method], fontsize=11)
                continue
            heatmap = np.zeros((2, d))
            for c in range(2):
                rpos  = rd[c]
                total = len(rpos)
                for r in range(1, d + 1):
                    heatmap[c, r - 1] = np.sum(rpos == r) / total
            ax.imshow(heatmap, aspect='auto', cmap='YlOrRd',
                      vmin=0, vmax=min(heatmap.max(), 1.0))
            for ci in range(2):
                for ri in range(d):
                    val = heatmap[ci, ri]
                    if val > 0.05:
                        txt_color = 'white' if val > 0.55 else 'black'
                        ax.text(ri, ci, f'{val:.0%}', ha='center', va='center',
                                fontsize=7, color=txt_color)
            ax.set_xticks(range(d))
            ax.set_xticklabels([str(r + 1) for r in range(d)], fontsize=8)
            ax.set_yticks([0, 1])
            ax.set_yticklabels(['Avg #1', 'Avg #2'], fontsize=9)
            ax.set_xlabel('Rank in individual seed', fontsize=9)
            ax.set_title(METHOD_LABELS[method], fontsize=11)
        fig.tight_layout()
        fname = os.path.join(FIG_DIR, f'churn_rankpos_k2_{ds}_{model}.pdf')
        fig.savefig(fname, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')

## ══════════════ Top-4 Jaccard ══════════════

In [ ]:
# Config and discrete Jaccard proportions for top-4
_jv4 = [0.0, 0.4, round(2/3, 6), round(6/7, 6), 1.0]
_cmap_obj = plt.cm.RdYlGn_r
_colors4 = {v: _cmap_obj(t) for v, t in zip(_jv4, np.linspace(0.05, 0.95, len(_jv4)))}
_labels4 = {
    0.0:           'Jac=0 (4 feat. overlap)',
    0.4:           'Jac=0.4 (3 feat. overlap)',
    round(2/3, 6): 'Jac=0.667 (2 feat. overlap)',
    round(6/7, 6): 'Jac=0.857 (1 feat. overlap)',
    1.0:           'Jac=1.0 (0 feat. overlap)',
}

jac_props_k4 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue
            min_n = min(len(v) for v in svs)
            all_jac = np.concatenate([
                compute_jaccard_distance(svs[i][:min_n], svs[j][:min_n], k=4)
                for i, j in PAIRS_10
            ])
            counts = {v: 0 for v in _jv4}
            for val in all_jac:
                counts[round_jac_k(float(val), _jv4)] += 1
            total = len(all_jac)
            jac_props_k4[ds][model][method] = {v: counts[v] / total for v in _jv4}

print('Top-4 Jaccard proportions computed.')

In [ ]:
# Top-4 discrete Jaccard stacked bar charts — 1 figure per dataset (1x2: mlp, ftt)
for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Discrete Top-4 Jaccard Distribution  (bg_size=100)',
        fontsize=13, fontweight='bold'
    )
    for ax, model in zip(axes, MODELS):
        avail = [m for m in METHODS if m in jac_props_k4[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue
        x       = np.arange(len(avail))
        bottoms = np.zeros(len(avail))
        for jv in _jv4:
            heights = np.array([jac_props_k4[ds][model][m][jv] for m in avail])
            ax.bar(x, heights, bottom=bottoms,
                   color=_colors4[jv], edgecolor='white', linewidth=0.5)
            for xi, (h, b) in enumerate(zip(heights, bottoms)):
                if h > 0.04:
                    txt_color = 'white' if jv == _jv4[-1] else 'black'
                    ax.text(xi, b + h / 2, f'{h:.2%}',
                            ha='center', va='center', fontsize=8,
                            color=txt_color, fontweight='bold')
            bottoms += heights
        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, 1.02)
        ax.set_ylabel('Proportion of (pair, instance) observations')
        ax.set_title(model.upper())
    handles = [mpatches.Patch(color=_colors4[jv], label=_labels4[jv]) for jv in _jv4]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9,
               bbox_to_anchor=(0.5, -0.10), framealpha=0.9)
    fig.tight_layout(rect=[0, 0.14, 1, 1])
    fname = os.path.join(FIG_DIR, f'churn_jaccard_k4_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Top-4 pairwise churn: % of (pair, instance) where top-4 SET changes
# Derived directly from jac_props_k4: churn = 1 - prop(Jac=0)
topk_pairchurn_k4 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            if method not in jac_props_k4[ds][model]:
                continue
            topk_pairchurn_k4[ds][model][method] = 1.0 - jac_props_k4[ds][model][method][0.0]

for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Top-4 Pairwise Churn  (bg_size=100)',
        fontsize=13, fontweight='bold'
    )
    for ax, model in zip(axes, MODELS):
        avail = [m for m in METHODS if m in topk_pairchurn_k4[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue
        x      = np.arange(len(avail))
        values = [topk_pairchurn_k4[ds][model][m] for m in avail]
        colors = [METHOD_COLORS[m] for m in avail]
        ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5)
        for xi, v in enumerate(values):
            ax.text(xi, v + 0.005, f'{v:.2%}', ha='center', va='bottom', fontsize=9)
        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, max(values) * 1.30 + 0.01)
        ax.set_ylabel('% of pairs where top-4 set changes')
        ax.set_title(model.upper())
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f'churn_topk_pair_k4_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Top-4 any-seed churn: % of instances where top-4 SET differs across any seed
topk_anychurn_k4 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue
            min_n = min(len(v) for v in svs)
            # Sort feature indices so set comparison is order-independent
            topk_sorted = np.stack([
                np.sort(np.argsort(-np.abs(svs[s][:min_n]), axis=1)[:, :4], axis=1)
                for s in range(N_BG_SEEDS)
            ], axis=0)                                           # (5, N, 4)
            all_agree = np.all(topk_sorted == topk_sorted[[0], :, :], axis=0)  # (N, 4)
            stable = np.all(all_agree, axis=1)                  # (N,)
            topk_anychurn_k4[ds][model][method] = float((~stable).mean())

for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Top-4 Any-Seed Churn  (bg_size=100)\n'
        r'(% instances where $\geq$1 seed has a different top-4 set)',
        fontsize=12, fontweight='bold'
    )
    for ax, model in zip(axes, MODELS):
        avail = [m for m in METHODS if m in topk_anychurn_k4[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue
        x      = np.arange(len(avail))
        values = [topk_anychurn_k4[ds][model][m] for m in avail]
        colors = [METHOD_COLORS[m] for m in avail]
        ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5)
        for xi, v in enumerate(values):
            ax.text(xi, v + 0.005, f'{v:.2%}', ha='center', va='bottom', fontsize=9)
        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, max(values) * 1.30 + 0.01)
        ax.set_ylabel('% of instances where top-4 set changes')
        ax.set_title(model.upper())
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f'churn_topk_any_k4_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Rank position distribution for consensus top-4
rank_dist_k4 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue
            min_n   = min(len(v) for v in svs)
            abs_svs = np.stack([np.abs(v[:min_n]) for v in svs], axis=0)  # (5, N, F)
            mean_abs       = abs_svs.mean(axis=0)
            consensus_topk = np.argsort(-mean_abs, axis=1)[:, :4]         # (N, 4)
            ranks = np.argsort(np.argsort(-abs_svs, axis=2), axis=2)      # (5, N, F)
            rank_dist_k4[ds][model][method] = {}
            for c in range(4):
                feat_idx = consensus_topk[:, c]
                rpos = np.array([
                    ranks[s, np.arange(min_n), feat_idx]
                    for s in range(N_BG_SEEDS)
                ]).flatten()
                rank_dist_k4[ds][model][method][c] = rpos + 1
print('Top-4 rank distributions computed.')

In [ ]:
# Top-4 rank position heatmaps — 1 figure per (dataset, model), 1x4 subplots
for ds in DATASETS:
    d = N_FEATS[ds]
    for model in MODELS:
        fig, axes = plt.subplots(1, 4, figsize=(14, 4.0))
        fig.suptitle(
            f'{DATASET_LABELS[ds]}  ·  {model.upper()}  —  Rank Positions of Consensus Top-4 Features  (bg_size=100)',
            fontsize=12, fontweight='bold'
        )
        for col, method in enumerate(METHODS):
            ax = axes[col]
            rd = rank_dist_k4[ds][model].get(method)
            if rd is None:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                        transform=ax.transAxes, fontsize=12, color='gray')
                ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(METHOD_LABELS[method], fontsize=11)
                continue
            heatmap = np.zeros((4, d))
            for c in range(4):
                rpos  = rd[c]
                total = len(rpos)
                for r in range(1, d + 1):
                    heatmap[c, r - 1] = np.sum(rpos == r) / total
            ax.imshow(heatmap, aspect='auto', cmap='YlOrRd',
                      vmin=0, vmax=min(heatmap.max(), 1.0))
            for ci in range(4):
                for ri in range(d):
                    val = heatmap[ci, ri]
                    if val > 0.05:
                        txt_color = 'white' if val > 0.55 else 'black'
                        ax.text(ri, ci, f'{val:.0%}', ha='center', va='center',
                                fontsize=7, color=txt_color)
            ax.set_xticks(range(d))
            ax.set_xticklabels([str(r + 1) for r in range(d)], fontsize=8)
            ax.set_yticks(range(4))
            ax.set_yticklabels([f'Avg #{c+1}' for c in range(4)], fontsize=9)
            ax.set_xlabel('Rank in individual seed', fontsize=9)
            ax.set_title(METHOD_LABELS[method], fontsize=11)
        fig.tight_layout()
        fname = os.path.join(FIG_DIR, f'churn_rankpos_k4_{ds}_{model}.pdf')
        fig.savefig(fname, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')

## ══════════════ Top-5 Jaccard ══════════════

In [ ]:
# Config and discrete Jaccard proportions for top-5
_jv5 = [0.0, round(1/3, 6), round(4/7, 6), 0.75, round(8/9, 6), 1.0]
_cmap_obj = plt.cm.RdYlGn_r
_colors5 = {v: _cmap_obj(t) for v, t in zip(_jv5, np.linspace(0.05, 0.95, len(_jv5)))}
_labels5 = {
    0.0:           'Jac=0 (5 feat. overlap)',
    round(1/3, 6): 'Jac=0.333 (4 feat. overlap)',
    round(4/7, 6): 'Jac=0.571 (3 feat. overlap)',
    0.75:          'Jac=0.75 (2 feat. overlap)',
    round(8/9, 6): 'Jac=0.889 (1 feat. overlap)',
    1.0:           'Jac=1.0 (0 feat. overlap)',
}

jac_props_k5 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue
            min_n = min(len(v) for v in svs)
            all_jac = np.concatenate([
                compute_jaccard_distance(svs[i][:min_n], svs[j][:min_n], k=5)
                for i, j in PAIRS_10
            ])
            counts = {v: 0 for v in _jv5}
            for val in all_jac:
                counts[round_jac_k(float(val), _jv5)] += 1
            total = len(all_jac)
            jac_props_k5[ds][model][method] = {v: counts[v] / total for v in _jv5}

print('Top-5 Jaccard proportions computed.')

In [ ]:
# Top-5 discrete Jaccard stacked bar charts — 1 figure per dataset (1x2: mlp, ftt)
for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Discrete Top-5 Jaccard Distribution  (bg_size=100)',
        fontsize=13, fontweight='bold'
    )
    for ax, model in zip(axes, MODELS):
        avail = [m for m in METHODS if m in jac_props_k5[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue
        x       = np.arange(len(avail))
        bottoms = np.zeros(len(avail))
        for jv in _jv5:
            heights = np.array([jac_props_k5[ds][model][m][jv] for m in avail])
            ax.bar(x, heights, bottom=bottoms,
                   color=_colors5[jv], edgecolor='white', linewidth=0.5)
            for xi, (h, b) in enumerate(zip(heights, bottoms)):
                if h > 0.04:
                    txt_color = 'white' if jv == _jv5[-1] else 'black'
                    ax.text(xi, b + h / 2, f'{h:.2%}',
                            ha='center', va='center', fontsize=8,
                            color=txt_color, fontweight='bold')
            bottoms += heights
        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, 1.02)
        ax.set_ylabel('Proportion of (pair, instance) observations')
        ax.set_title(model.upper())
    handles = [mpatches.Patch(color=_colors5[jv], label=_labels5[jv]) for jv in _jv5]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9,
               bbox_to_anchor=(0.5, -0.10), framealpha=0.9)
    fig.tight_layout(rect=[0, 0.14, 1, 1])
    fname = os.path.join(FIG_DIR, f'churn_jaccard_k5_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Top-5 pairwise churn: % of (pair, instance) where top-5 SET changes
# Derived directly from jac_props_k5: churn = 1 - prop(Jac=0)
topk_pairchurn_k5 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            if method not in jac_props_k5[ds][model]:
                continue
            topk_pairchurn_k5[ds][model][method] = 1.0 - jac_props_k5[ds][model][method][0.0]

for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Top-5 Pairwise Churn  (bg_size=100)',
        fontsize=13, fontweight='bold'
    )
    for ax, model in zip(axes, MODELS):
        avail = [m for m in METHODS if m in topk_pairchurn_k5[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue
        x      = np.arange(len(avail))
        values = [topk_pairchurn_k5[ds][model][m] for m in avail]
        colors = [METHOD_COLORS[m] for m in avail]
        ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5)
        for xi, v in enumerate(values):
            ax.text(xi, v + 0.005, f'{v:.2%}', ha='center', va='bottom', fontsize=9)
        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, max(values) * 1.30 + 0.01)
        ax.set_ylabel('% of pairs where top-5 set changes')
        ax.set_title(model.upper())
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f'churn_topk_pair_k5_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Top-5 any-seed churn: % of instances where top-5 SET differs across any seed
topk_anychurn_k5 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue
            min_n = min(len(v) for v in svs)
            # Sort feature indices so set comparison is order-independent
            topk_sorted = np.stack([
                np.sort(np.argsort(-np.abs(svs[s][:min_n]), axis=1)[:, :5], axis=1)
                for s in range(N_BG_SEEDS)
            ], axis=0)                                           # (5, N, 5)
            all_agree = np.all(topk_sorted == topk_sorted[[0], :, :], axis=0)  # (N, 5)
            stable = np.all(all_agree, axis=1)                  # (N,)
            topk_anychurn_k5[ds][model][method] = float((~stable).mean())

for ds in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(
        f'{DATASET_LABELS[ds]}  —  Top-5 Any-Seed Churn  (bg_size=100)\n'
        r'(% instances where $\geq$1 seed has a different top-5 set)',
        fontsize=12, fontweight='bold'
    )
    for ax, model in zip(axes, MODELS):
        avail = [m for m in METHODS if m in topk_anychurn_k5[ds][model]]
        if not avail:
            ax.set_visible(False)
            continue
        x      = np.arange(len(avail))
        values = [topk_anychurn_k5[ds][model][m] for m in avail]
        colors = [METHOD_COLORS[m] for m in avail]
        ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5)
        for xi, v in enumerate(values):
            ax.text(xi, v + 0.005, f'{v:.2%}', ha='center', va='bottom', fontsize=9)
        ax.set_xticks(x)
        ax.set_xticklabels([METHOD_LABELS[m] for m in avail])
        ax.set_ylim(0, max(values) * 1.30 + 0.01)
        ax.set_ylabel('% of instances where top-5 set changes')
        ax.set_title(model.upper())
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f'churn_topk_any_k5_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# Rank position distribution for consensus top-5
rank_dist_k5 = {ds: {mo: {} for mo in MODELS} for ds in DATASETS}
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            svs = sv_data[ds][model][method]
            if svs is None:
                continue
            min_n   = min(len(v) for v in svs)
            abs_svs = np.stack([np.abs(v[:min_n]) for v in svs], axis=0)  # (5, N, F)
            mean_abs       = abs_svs.mean(axis=0)
            consensus_topk = np.argsort(-mean_abs, axis=1)[:, :5]         # (N, 5)
            ranks = np.argsort(np.argsort(-abs_svs, axis=2), axis=2)      # (5, N, F)
            rank_dist_k5[ds][model][method] = {}
            for c in range(5):
                feat_idx = consensus_topk[:, c]
                rpos = np.array([
                    ranks[s, np.arange(min_n), feat_idx]
                    for s in range(N_BG_SEEDS)
                ]).flatten()
                rank_dist_k5[ds][model][method][c] = rpos + 1
print('Top-5 rank distributions computed.')

In [ ]:
# Top-5 rank position heatmaps — 1 figure per (dataset, model), 1x4 subplots
for ds in DATASETS:
    d = N_FEATS[ds]
    for model in MODELS:
        fig, axes = plt.subplots(1, 4, figsize=(14, 4.5))
        fig.suptitle(
            f'{DATASET_LABELS[ds]}  ·  {model.upper()}  —  Rank Positions of Consensus Top-5 Features  (bg_size=100)',
            fontsize=12, fontweight='bold'
        )
        for col, method in enumerate(METHODS):
            ax = axes[col]
            rd = rank_dist_k5[ds][model].get(method)
            if rd is None:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                        transform=ax.transAxes, fontsize=12, color='gray')
                ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(METHOD_LABELS[method], fontsize=11)
                continue
            heatmap = np.zeros((5, d))
            for c in range(5):
                rpos  = rd[c]
                total = len(rpos)
                for r in range(1, d + 1):
                    heatmap[c, r - 1] = np.sum(rpos == r) / total
            ax.imshow(heatmap, aspect='auto', cmap='YlOrRd',
                      vmin=0, vmax=min(heatmap.max(), 1.0))
            for ci in range(5):
                for ri in range(d):
                    val = heatmap[ci, ri]
                    if val > 0.05:
                        txt_color = 'white' if val > 0.55 else 'black'
                        ax.text(ri, ci, f'{val:.0%}', ha='center', va='center',
                                fontsize=7, color=txt_color)
            ax.set_xticks(range(d))
            ax.set_xticklabels([str(r + 1) for r in range(d)], fontsize=8)
            ax.set_yticks(range(5))
            ax.set_yticklabels([f'Avg #{c+1}' for c in range(5)], fontsize=9)
            ax.set_xlabel('Rank in individual seed', fontsize=9)
            ax.set_title(METHOD_LABELS[method], fontsize=11)
        fig.tight_layout()
        fname = os.path.join(FIG_DIR, f'churn_rankpos_k5_{ds}_{model}.pdf')
        fig.savefig(fname, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')